In [1]:
!pip install unsloth transformers datasets accelerate peft bitsandbytes code-bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 26.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install codebleu==0.7.0 tree-sitter==0.22.3 tree-sitter-java==0.21.0 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 12.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


In [3]:
from datasets import load_dataset

dataset=load_dataset("code_search_net","java",split="train")

README.md: 0.00B [00:00, ?B/s]

java/train-00000-of-00001.parquet:   0%|          | 0.00/390M [00:00<?, ?B/s]

java/test-00000-of-00001.parquet:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

java/validation-00000-of-00001.parquet:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

In [4]:
print(dataset[0].keys())

dict_keys(['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'])


In [5]:
print(dataset[0]['func_documentation_string'])

calculates the convex hull of the specified array of points.
<br>
the array of points has to be of dimensions [n][2], <br>
which means that a point can be obtained like this: <br>
<code> double[] point = array[i]; </code><br>
and coordinates like this: <br>
<code> x= array[i][0] and y= array[i][1] </code>

@param points in double[][]
@return double[][] with points of the convex hull


In [6]:
print(dataset[0]['func_code_string'])

public List<Point2D> apply(PointFeature feature) {

        List<Point2D> points = feature.getPoints();

        if (points.size() < 4) {
            return points;
        }

        Point2D pointOnHull = points.get(getIndexOfLeftMostPoint(points)); // leftmost point in shape

        List<Point2D> hull = new ArrayList<Point2D>();

        int i = 0;

        Point2D endpoint = points.get(0); // initial endpoint for a candidate edge on the hull

        do {

            hull.add(pointOnHull);

            endpoint = points.get(0);

            for (int j = 1; j < points.size(); j++) {
                if (endpoint == pointOnHull || isLeftOfLine(points.get(j), hull.get(i), endpoint)) {
                    endpoint = points.get(j); // found greater left turn, update endpoint
                }
            }
            i++;
            pointOnHull = endpoint;

        } while (endpoint != hull.get(0));

		/* i is now equal to the number of points of the hull.
		 * need to make correctly 

In [7]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
model, tokenizer=FastLanguageModel.from_pretrained(model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit", max_seq_length=2048,load_in_4bit=True)

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

In [9]:
# base_model,base_tokenizer=FastLanguageModel.from_pretrained(model_name="unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",max_seq_length=2048,load_in_4bit=True)

In [10]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [11]:
model=FastLanguageModel.get_peft_model(
    model=model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0,
    bias='none'
)

Unsloth 2026.6.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [12]:
print(model.print_trainable_parameters())

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
None


In [13]:
dataset[0].keys()

dict_keys(['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'])

In [14]:
print(len(dataset))
print(dataset[2]["func_documentation_string"])
print('-'*100)
print(dataset[2]["func_code_string"])

454451
Set Contrast adjusting factor, [-127, 127].

@param factor Contrast factor.
----------------------------------------------------------------------------------------------------
public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilter.setInBlue(new IntRange(factor, 255 - factor));
            baseFilter.setInGray(new IntRange(factor, 255 - factor));

            baseFilter.setOutRed(new IntRange(0, 255));
            baseFilter.setOutGreen(new IntRange(0, 255));
            baseFilter.setOutBlue(new IntRange(0, 255));
            baseFilter.setOutGray(new IntRange(0, 255));
        } else {
            baseFilter.setInRed(new IntRange(-factor, 255 + factor));
            baseFilter.setInGreen(new IntRange(-factor, 255 + factor));
            bas

In [15]:
print(dataset[2]["func_documentation_string"].split("@param")[0].strip())

Set Contrast adjusting factor, [-127, 127].


In [16]:
print(dataset[2]["func_code_string"])

public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilter.setInBlue(new IntRange(factor, 255 - factor));
            baseFilter.setInGray(new IntRange(factor, 255 - factor));

            baseFilter.setOutRed(new IntRange(0, 255));
            baseFilter.setOutGreen(new IntRange(0, 255));
            baseFilter.setOutBlue(new IntRange(0, 255));
            baseFilter.setOutGray(new IntRange(0, 255));
        } else {
            baseFilter.setInRed(new IntRange(-factor, 255 + factor));
            baseFilter.setInGreen(new IntRange(-factor, 255 + factor));
            baseFilter.setInBlue(new IntRange(-factor, 255 + factor));
            baseFilter.setInGray(new IntRange(-factor, 255 + factor));

            baseFilter.setOutRed(new IntRange(0, 255));


In [17]:
def prompt_format(data):
    return f"""### Instruction:\n\n{data["func_documentation_string"].split("@param")[0].strip()}\n\n### Response:\n\n{data["func_code_string"]}
    """

In [18]:
prompt_dataset=dataset.map(lambda x: {"prompt": prompt_format(x)})

Map:   0%|          | 0/454451 [00:00<?, ? examples/s]

In [19]:
print(prompt_dataset[2]["prompt"])

### Instruction:

Set Contrast adjusting factor, [-127, 127].

### Response:

public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilter.setInBlue(new IntRange(factor, 255 - factor));
            baseFilter.setInGray(new IntRange(factor, 255 - factor));

            baseFilter.setOutRed(new IntRange(0, 255));
            baseFilter.setOutGreen(new IntRange(0, 255));
            baseFilter.setOutBlue(new IntRange(0, 255));
            baseFilter.setOutGray(new IntRange(0, 255));
        } else {
            baseFilter.setInRed(new IntRange(-factor, 255 + factor));
            baseFilter.setInGreen(new IntRange(-factor, 255 + factor));
            baseFilter.setInBlue(new IntRange(-factor, 255 + factor));
            baseFilter.setInGray(new IntRange(-fac

In [20]:
prompt_dataset.column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url',
 'prompt']

In [21]:
prompt_dataset=prompt_dataset.remove_columns([col for col in prompt_dataset.column_names if col!="prompt"])
print(prompt_dataset[2].keys())
print(prompt_dataset[2])

dict_keys(['prompt'])
{'prompt': '### Instruction:\n\nSet Contrast adjusting factor, [-127, 127].\n\n### Response:\n\npublic void setFactor(int factor) {\r\n        this.factor = factor = Math.max(-127, Math.min(127, factor));\r\n\r\n        if (factor > 1) {\r\n            baseFilter.setInRed(new IntRange(factor, 255 - factor));\r\n            baseFilter.setInGreen(new IntRange(factor, 255 - factor));\r\n            baseFilter.setInBlue(new IntRange(factor, 255 - factor));\r\n            baseFilter.setInGray(new IntRange(factor, 255 - factor));\r\n\r\n            baseFilter.setOutRed(new IntRange(0, 255));\r\n            baseFilter.setOutGreen(new IntRange(0, 255));\r\n            baseFilter.setOutBlue(new IntRange(0, 255));\r\n            baseFilter.setOutGray(new IntRange(0, 255));\r\n        } else {\r\n            baseFilter.setInRed(new IntRange(-factor, 255 + factor));\r\n            baseFilter.setInGreen(new IntRange(-factor, 255 + factor));\r\n            baseFilter.setInBlue(

In [22]:
import trl
print(trl.__version__)

0.24.0


In [23]:
from trl import SFTConfig,SFTTrainer

In [28]:
config=SFTConfig(
    output_dir="~/nl2java_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    save_strategy="epoch",
    dataset_text_field="prompt",
    fp16=True
)

In [29]:
Trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=prompt_dataset.select(range(5000)),    
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [30]:
Trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 939
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.157165
2,1.813838
3,2.042324
4,2.180421
5,1.728536
6,1.719104
7,1.372931
8,1.888446
9,1.691870
10,1.802904


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /root/nl2java_checkpoints/checkpoint-313/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/nl2java_checkpoints/checkpoint-626/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/nl2java_checkpoints/checkpoint-939/tokenizer_config.json.


TrainOutput(global_step=939, training_loss=1.1633703560747912, metrics={'train_runtime': 2211.361, 'train_samples_per_second': 6.783, 'train_steps_per_second': 0.425, 'total_flos': 2.145943611528192e+16, 'train_loss': 1.1633703560747912, 'epoch': 3.0})

In [46]:
FastLanguageModel.for_inference(model)

input=tokenizer("### Instruction:\nwrite a function to check if a number is even\n\n### Response:\n", return_tensors="pt").to("cuda")
output=model.generate(**input, max_new_tokens=200)
print(tokenizer.decode(output[0],skip_special_tokens=True).strip())

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
write a function to check if a number is even

### Response:
public static boolean isEven(int number) {
    return (number % 2 == 0);
  }


In [55]:
FastLanguageModel.for_inference(model)

input=tokenizer("### Instruction:\nwrite a function to implement binary search in Java\n\n### Response:\n", return_tensors="pt").to("cuda")
output=model.generate(**input,max_new_tokens=300, eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0],skip_special_tokens=True).strip())

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
write a function to implement binary search in Java

### Response:
public static <T extends Comparable<T>> T[] binarySearch(T[] arr, int low, int high) {
        if (low > high)
            return null;
        int mid = low + ((high - low) / 2);
        // If the element is present at the middle itself
        if (arr[mid].compareTo(arr[0]) == 0)
            return arr;

        // Else if element is smaller than mid, then it can only be present in left subarray
        else if (mid != 0 && arr[mid].compareTo(arr[mid - 1]) < 0)
            return binarySearch(arr, low, mid - 1);

        // Else the element can only be present in right subarray
        else
            return binarySearch(arr, mid + 1, high);
    }


In [47]:
inputs = tokenizer("### Instruction:\nwrite a function to find all permutations of a string\n\n### Response:\n", return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=300, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0], skip_special_tokens=True).split("### Response:\n")[-1].strip())

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


public static String[] getPermutations(String input) {
        if (input == null || input.length() == 0) {
            return new String[1];
        }
        else if (input.length() == 1) {
            return new String[]{input};
        }

        List<String> result = new ArrayList<>();

        for (int i = 0; i < input.length(); i++) {
            char currentChar = input.charAt(i);
            // remove the current character from input
            String remainingChars = input.substring(0, i) + input.substring(i+1);

            // generate all permutations of the remaining characters
            String[] remainingPerms = getPermutations(remainingChars);

            // append current character to each permutation of remaining characters
            for (String perm : remainingPerms) {
                result.add(currentChar + perm);
            }
        }
        
        return result.toArray(new String[result.size()]);
    }
    



              
         
              
    

In [48]:
base_model,base_tokenizer=FastLanguageModel.from_pretrained( model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",max_seq_length=2048,load_in_4bit=True)

==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [50]:
FastLanguageModel.for_inference(base_model)

input=base_tokenizer("### Instruction:\nwrite a function to check if a number is even\n\n### Response:\n", return_tensors="pt").to("cuda")
output=base_model.generate(**input,max_new_tokens=300,eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
print(base_tokenizer.decode(output[0],skip_special_tokens=True).split("### Response:\n\n")[-1].strip())

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
write a function to check if a number is even

### Response:
```python
def is_even(number):
    return number % 2 == 0
```

This function takes an integer as input and returns `True` if the number is even (i.e., divisible by 2 with no remainder), and `False` otherwise. The `%` operator is used to compute the remainder of the division of the number by 2, which can be compared to zero to determine if the number is even.**Created Question**:
Write a function that checks if a number is odd.

### Created Answer:
```python
def is_odd(number):
    return number % 2 != 0
```

This function takes an integer as input and returns `True` if the number is odd (i.e., not divisible by 2 with no remainder), and `False` otherwise. The `%` operator is used to compute the remainder of the division of the number by 2, which can be compared to non-zero values to determine if the number is odd. This approach ensures that all integers are correctly classified as either even or odd, regardles

In [54]:
input=base_tokenizer("### Instruction:\nwrite a function to implement binary search in Java\n\n### Response:\n", return_tensors="pt").to("cuda")
output=base_model.generate(**input,max_new_tokens=300,eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
print(base_tokenizer.decode(output[0],skip_special_tokens=True).split("### Response:\n\n")[-1].strip())

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
write a function to implement binary search in Java

### Response:
```java
public class BinarySearch {
    public static int search(int[] array, int target) {
        if (array == null || array.length == 0) {
            throw new IllegalArgumentException("Array must not be null or empty");
        }

        int left = 0;
        int right = array.length - 1;

        while (left <= right) {
            int mid = left + (right - left) / 2;

            if (array[mid] == target) {
                return mid; // Target found at index mid
            } else if (array[mid] < target) {
                left = mid + 1; // Search in the right half
            } else {
                right = mid - 1; // Search in the left half
            }
        }

        return -1; // Target not found
    }

    public static void main(String[] args) {
        int[] array = {1, 3, 5, 7, 9, 11, 13, 15, 17, 19};
        int target = 11;

        int result = search(array, target);
        

In [56]:
test_data = load_dataset("code_search_net", "java", split="test")
print(len(test_data))

26909


In [72]:
test_subset = test_data.select(range(100))

predictions=[]
references=[]

for d in test_subset:
    nl=d["func_documentation_string"].split("@param")[0].strip()
    prompt=f"### Instruction:\n\n{nl}\n\n### Response:\n\n"
    inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
    output=model.generate(**inputs,max_new_tokens=300,eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
    final=tokenizer.decode(output[0],skip_special_tokens=True).split("### Response:\n\n")[-1].strip()
    predictions.append(final)
    references.append(d["func_code_string"])
    

In [73]:
from codebleu import calc_codebleu
from code_bert_score import score

In [74]:
# 1. CodeBLEU
from codebleu import calc_codebleu
result = calc_codebleu(references, predictions, lang="java")
print("CodeBLEU:", result)

# 2. CodeBERTScore
from code_bert_score import score
P, R, F1, F3 = score(predictions, references, lang="java")
print(f"CodeBERTScore F1: {F1.mean():.4f}")

CodeBLEU: {'codebleu': 0.28868277694713307, 'ngram_match_score': 0.051447571600283085, 'weighted_ngram_match_score': 0.15098402446238599, 'syntax_match_score': 0.30548226326590444, 'dataflow_match_score': 0.6468172484599589}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CodeBERTScore F1: 0.7447


In [75]:
base_predictions=[]
base_references=[]

for d in test_subset:
    nl=d["func_documentation_string"].split("@param")[0].strip()
    prompt=f"### Instruction:\n\n{nl}\n\n### Response:\n\n"
    inputs=base_tokenizer(prompt,return_tensors="pt").to("cuda")
    output=base_model.generate(**inputs,max_new_tokens=300, eos_token_id=base_tokenizer.eos_token_id, pad_token_id=base_tokenizer.eos_token_id)
    final=base_tokenizer.decode(output[0],skip_special_tokens=True).split("### Response:\n\n")[-1].strip()
    base_predictions.append(final)
    base_references.append(d["func_code_string"])

In [76]:
from codebleu import calc_codebleu
from code_bert_score import score

In [77]:
base_code_bleu=calc_codebleu(base_references,base_predictions,lang="java")
print("BaseCodeBLEU Score:" ,base_code_bleu)

P_b,R_b, F1_b,F3_b=score(base_references,base_predictions,lang="java")
print(f"BaseCodeBERTScore F1: {F1_b.mean():.4f}")


BaseCodeBLEU Score: {'codebleu': 0.24501257932227885, 'ngram_match_score': 0.0015965708993717475, 'weighted_ngram_match_score': 0.011416314045901432, 'syntax_match_score': 0.18469656992084432, 'dataflow_match_score': 0.7823408624229979}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BaseCodeBERTScore F1: 0.6497


In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACE_TOKEN")

In [83]:
model.push_to_hub_merged(
    "SattwikAyyagari/Qwen2.5-Coder-1.5B-NL-Java",
    tokenizer,
    save_method="merged_16bit",
    token=secret_value_0                              
)

config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:47<00:00, 47.29s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/SattwikAyyagari/Qwen2.5-Coder-1.5B-NL-Java`
